# Project: Reimplementation and Improvement of the RLT Algorithm (Reinforcement Learning Trees)

**Goal:** This notebook structures the project for reimplementing the RLT algorithm in Python.

**Authors:** Kousay Najar, Hamza Farhani, Taoufik Krid, Wiem Ben M’Sahel, Rawen Mezzi, Mohamed Khayat

**Date:** 11/2025

## Phase 1: Business Understanding

This phase aims to define the objectives from a business perspective and translate them into a well-defined data science problem.

# Problem statement & Study of Existing Methods and Limitations & Defining Strategy

### 1. Problem Statement

#### Context and Motivation
The problem addresses high-dimensional sparse settings where traditional tree-based methods like Random Forests show limitations. In scenarios with $p$ variables, only $p_1 \ll p$ strong variables carry the true signal, while $p_2 = p - p_1$ are noise variables.

#### Key Challenges Identified
**Random Feature Selection Limitations:** In high-dimensional settings with many noise variables, random feature selection provides little opportunity to consider strong variables as splitting rules. When $p$ is large and $p_1$ is small, the probability of selecting a strong variable decreases dramatically.

**Terminal Node Degradation:** As sample size decreases toward terminal nodes, identifying important variables becomes increasingly difficult regardless of the model used. This causes splitting variable selection to behave almost randomly, leading to performance similar to purely random forests.

**Hidden Structures:** Marginal comparisons of splitting variables can fail to identify strong variables, especially with structures like the checkerboard pattern where variables show little marginal effect but strong joint effects.

#### Research Objectives
Develop a tree-based method that achieves consistency with convergence rates depending only on $p_1$ (number of strong variables) rather than $p$ (total number of variables). The method should force splits to concentrate on strong variables throughout the tree construction, especially toward terminal nodes.

### 2. Study of Existing Methods and Limitations

#### Traditional Random Forests
**Strengths:** State-of-the-art ensemble method with flexible non-parametric structure and capacity for handling high-dimensional data. Shows great potential in cancer studies with large numbers of genes or SNPs.

**Limitations:**
- Unsatisfactory performance in some studies compared to other machine learning tools
- Random feature selection creates bias in variable importance measures when using small numbers of features
- Using large numbers of predictors causes overfitting toward terminal nodes where sample size is small
- Lack of theoretical attention on sparsity for tree-based methods

#### Alternative Tree-Based Methods
- **Extremely Randomized Trees (ET):** Use random cut points rather than searching for best cut points, achieving similar performance to Random Forests at reduced computational cost.
- **Bayesian Additive Regression Trees (BART):** Integrate tree-based methods into a Bayesian framework.
- **Purely Random Forests:** Provide friendly framework for theoretical analysis but are extremely inefficient because most splits select noise variables, especially in sparse settings.

#### Linear Models
- **Lasso and Penalized Methods:** Among the most popular methods for identifying signal variables in linear models. However, they cannot capture complex non-linear and interaction effects that tree-based methods can handle.

#### Theoretical Gaps
- **Consistency Issues:** The asymptotic behavior of random forests relies heavily on the particular splitting rule implemented. Some greedy construction rules demonstrate inconsistency under certain conditions.
- **No Method with Both Properties:** Up to now, there appears to be no tree-based method possessing both established theoretical validity and excellent practical performance.

### 3. Defining Strategy

#### Three-Fold Innovation Approach

**1. Reinforcement Learning for Splitting (The "Look-Ahead")**
*   **Concept:** Instead of greedily choosing the split with the best *immediate* result (like standard Random Forests), RLT looks ahead.
*   **How:** It runs a small internal model (embedded model) at every node to calculate Variable Importance. It chooses variables that offer the best *future* rewards, allowing it to detect hidden patterns (like checkerboards) that standard trees miss.

**2. Progressive Variable Muting (The "Noise Filter")**
*   **Concept:** As the tree grows deeper and data becomes scarcer, the risk of splitting on noise increases.
*   **How:** RLT progressively "mutes" (discards) weak variables at each level. This forces deep nodes to split only on strong, proven signals, ensuring the model remains robust even with small sample sizes.

**3. Linear Combination Splits (The "Smarter Cut")**
*   **Concept:** Standard trees only cut horizontally or vertically (e.g., $X_1 > 5$).
*   **How:** RLT can split on a weighted combination of top variables (e.g., $0.5X_1 + 0.3X_2 > 0$). The weights are determined by the Variable Importance calculated in step 1, allowing the tree to capture local linear trends efficiently.

#### Theoretical & Validation Basis
*   **Theory:** The model is mathematically proven to converge based on the number of *strong* variables, effectively ignoring the total number of noise variables.
*   **Validation:** We validate this approach using:
    *   **4 Simulated Scenarios:** Testing linear, non-linear, and highly correlated relationships.
    *   **10 Real-World Datasets:** Benchmarking against Random Forests, Gradient Boosting, and Lasso on standard UCI datasets.

### 1.1 Business Objectives (BOs)
The BOs describe the expected value from a non-technical perspective.

- **BO1: Reimplement the strategy.** Define the strategy (RLT).  
- **BO2: Compare classical solutions with our solution.**  
- **BO3: Make the strategy's decisions explainable.**  
- **BO4: Optimize the strategy.**  

### 1.2 Data Science Objectives (DSOs)
DSOs are the technical goals that, once achieved, will fulfill the business objectives.

- **DSO1: Implement the Reinforcement Learning Trees (RLT) algorithm.**  
- **DSO2: Conduct a comparative study between the RLT model and classical methods.**  
- **DSO3: Explain the model's predictions using XAI techniques.**  
- **DSO4: Optimize the RLT model.**  


## Phase 2 : Data Understanding

We will analyze each dataset separately to understand their characteristics, distributions, and quality.

## Phase 3 : Préparation des Données (Data Preparation)

La préparation consistera principalement à diviser nos jeux de données synthétiques.

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")

for path in (project_root, src_path):
    if path not in sys.path:
        sys.path.append(path)


In [ ]:
import importlib
from utils import dataset_wrapper
from scripts import data_preparation

importlib.reload(dataset_wrapper)
importlib.reload(data_preparation)

"""for dataset_name in dataset_wrapper.datasets_dict.keys():
    wrapped_ds = dataset_wrapper.DatasetWrapper(dataset_name)
    _ = data_preparation.prepare_data(wrapped_ds)"""

'for dataset_name in dataset_wrapper.datasets_dict.keys():\n    wrapped_ds = dataset_wrapper.DatasetWrapper(dataset_name)\n    _ = data_preparation.prepare_data(wrapped_ds)'

### DSO4: Proposing and Testing Our Own Improvement

**Objective:** To validate if our proposed modifications to the RLT algorithm yield a statistically significant improvement in performance or efficiency over the original implementation.

| Model | Dataset(s) for Testing | Variables Involved | Key Parameters / Hyperparameters to Test |
| :--- | :--- | :--- | :--- |
| **RLT (Baseline)** | **All 10 real datasets** | **Identical pipeline for both models:**<br>1. Select numeric features only.<br>2. Standardize them (mean=0, var=1).<br>3. Add noisy covariates to reach **p=500**. | **Original paper's configuration:**<br>- `k` in<br>- `muting_rate` in [0, 0.5, 0.8]<br>- `embedded_model` = 'ExtremelyRandomizedTrees' |
| **Improved RLT** | **All 10 real datasets** | *Identical data pipeline as the baseline for a fair comparison.* | **Test one or more proposed improvements:**<br>- **Idea 1:** Change `embedded_model` to 'LightGBM'.<br>- **Idea 2:** Change `muting_strategy` to 'adaptive_quantile'.<br>- **Idea 3:** Change `linear_combination_method` to 'ridge_weighted'.<br><br>*(All other parameters remain identical to the baseline)* |

## Phase 5: Evaluation

Here, we validate the three DSOs separately.

## Phase 6: Deployment

Preparing the model for integration, with a focus on explainability (BO3).

### 6.1 Saving the Optimized Model
Save the best version of the model

### 6.2 Streamlit Interface (with XAI)
**Goal:** Interactive demo including transparency.

**Features:**
- User inputs  
- Display predictions  
- **Feature Heatmap (BO3):** Graphically show which variables influenced the decision (output from `.explain()`)  

### 6.3 FastAPI Endpoint
**Goal:** System integration.

**Endpoint:** `POST /predict`  
- **Input:** JSON data  
- **Output:** `{"prediction": value, "explanation": {feature_contributions}}`  
- The API returns not just the result, but also the reasoning behind it (BO3).  